# Multi-Sample scRNA-Seq IMPROVED Re-analysis Pipeline - pt2 bulk analysis

IMPROVED, PATCHED AND AUTOMATED Reanalysis of: 
* Melms, J.C., Biermann, J., Huang, H. et al. A molecular single-cell lung atlas of lethal COVID-19. Nature 595, 114–119 (2021). https://doi.org/10.1038/s41586-021-03569-1

    - https://www.nature.com/articles/s41586-021-03569-1

    - download the "RAW.tar" data. GEO with accession number GSE171524

  **INCLUDES:**
  - Automated functions for plotting umaps
  - optimized scvi layers for analysis and plotting (normalization only in required data)
  - Step by step manual cell labeling and Biological interpretation

### **by José Alberto Isidoro Sánchez**

##### GUIDE: https://youtu.be/uvyG9yLuNSE?si=fzfICWtsWN2JSAaj

In [ ]:
#conda install scanpy

----------------
----------------
----------------
# Overview
-----------------
-----------------
-----------------

This repository provides a multi-part, end-to-end single-cell RNA sequencing (scRNA-seq) reanalysis pipeline built around **scvi-tools** and **Scanpy** (reanalyzing GEO `GSE171524`, Melms et al., *Nature* 2021)[cite: 27, 28]. It spans per-sample doublet removal and quality control (**pt1**), memory-aware cohort integration, clustering, and cell-type annotation (**pt2**), and condition- or lineage-resolved differential expression and compositional analysis (**pt2** / **pt3**)[cite: 28, 170, 243, 254, 265].

## Pipeline Stages

1. **Per-Sample Preprocessing, Doublet Removal & QC (`pt1`)**
   - **Sample-by-Sample Execution**: Iterates over raw count matrices (`.csv.gz` / `.h5ad`) one sample at a time with `.h5ad` checkpointing and explicit memory cleanup (`del` + `gc.collect()`) to avoid RAM saturation[cite: 28, 83, 87].
   - **Deep Learning Doublet Detection**: Fits a sample-specific **scVI** VAE and trains a **SOLO** classifier on the learned latent space to score doublets (`dif = doublet - singlet`), filtering high-confidence doublets prior to downstream QC[cite: 28, 37, 42, 79].
   - **Cell & Gene QC Filtering**: Filters low-gene cells, annotates mitochondrial (`MT-`) and ribosomal (`KEGG_RIBOSOME`) genes, computes QC metrics on raw integer counts preserved in `adata.layers["counts"]`, and removes upper/outlier cells (`n_genes_by_counts` 98th percentile, `pct_counts_mt`, `pct_counts_ribo`) using `.copy()` to detach views[cite: 45, 47, 56, 79, 80].

2. **Cohort Concatenation, Integration & Cell-Type Annotation (`pt1` / `pt2`)**
   - **Memory-Safe Merging**: Loads purified per-sample `.h5ad` objects into CSR sparse matrices (`csr_matrix`), clears stale `uns`/`obsm` entries, concatenates in a single `ad.concat()` pass, deduplicates `.obs` columns, cleans `Sample` IDs/barcodes, and filters globally expressed genes[cite: 89, 92, 95].
   - **scVI Batch Integration**: Trains a cohort-level `SCVI` model on `layers["counts"]` with `Sample` as a categorical covariate and QC metrics (`pct_counts_mt`, `total_counts`, `pct_counts_ribo`) as continuous covariates, extracting the integrated latent representation (`X_scVI`) without storing dense `scvi_normalized` layers in RAM[cite: 122, 123].
   - **Clustering, Marker Discovery & Labeling**: Builds a neighbor graph on `X_scVI`, computes **UMAP** embeddings, explores **Leiden** clustering resolutions, combines frequentist (Wilcoxon) and scVI Bayesian differential expression with on-the-fly denoised plotting helpers, and maps manual lung cell-type annotations (`cell_type`)[cite: 131, 143, 166, 170, 236].

3. **Condition & Lineage Downstream Analysis (`pt3`)**
   - **Compositional Shifts**: Maps sample IDs to experimental conditions (`control` vs. `COVID19`) and quantifies per-sample and per-condition cell-type counts and proportions[cite: 241, 243, 244].
   - **Differential Expression & Response Profiling**: Runs scVI Bayesian differential expression across cell types (e.g., `AT1_Cells` vs. `AT2_Cells`), global conditions (`COVID19` vs. `control`), and within-cell-type condition comparisons (`cell_type_condition`) to inspect marker and injury-response gene programs[cite: 254, 263, 265].

## Key Architectural Notes

- **Why Doublets Are Detected Per Sample**: Droplet doublets reflect sample-specific cell loading and library preparation dynamics rather than a shared cohort property[cite: 28]. Training scVI/SOLO per sample and removing doublets before cohort concatenation and integration avoids cross-sample doublet misclassification[cite: 28].
- **Raw Counts vs. Normalized/Denoised Views**: Unnormalized integer counts are retained in `layers["counts"]` for scVI models, log-normalized expression is kept for standard visualization/Wilcoxon tests, and scVI denoised expression is queried on demand (or on small subsets) to keep `.h5ad` files and memory usage lean[cite: 120, 123, 147, 258].

-----
-----
-----
-----
-----
-----
-----
-----


# 4. BATCH PROCESSING AND MERGING (BEFORE INTEGRATION)


-----
-----
-----
-----
-----
-----
-----
-----

## 4.1 SETTING UP Batch Quality Control and Filtering

In [ ]:
import os
import pandas as pd
import numpy as np
import scanpy as sc
import scvi
from scipy.sparse import csr_matrix

def pp(csv_path, n_epocs):
    # =========================================================================
    # SECTION 1: DOUBLET DETECTION SETUP & TRAINING
    # =========================================================================
    adata = sc.read_csv(csv_path).T
    
    # Filter genes and identify HVGs for SOLO
    sc.pp.filter_genes(adata, min_cells=10)
    sc.pp.highly_variable_genes(adata, n_top_genes=2000, subset=True, flavor='seurat_v3')
    
    # Train scVI VAE & SOLO doublet classifier on MPS
    scvi.model.SCVI.setup_anndata(adata)
    vae = scvi.model.SCVI(adata)
    vae.train(
   
        max_epochs= n_epocs,
        early_stopping=True,
        early_stopping_patience=15,
        accelerator="mps"
    )
    
    solo = scvi.external.SOLO.from_scvi_model(vae)
    solo.train(
        max_epochs=n_epochs,
        early_stopping=True,
        early_stopping_patience=15,
        accelerator="mps"    
    )

    # =========================================================================
    # SECTION 2: DOUBLET PREDICTION & FILTERING THRESHOLDS
    # =========================================================================
    df = solo.predict()
    df['prediction'] = solo.predict(soft=False)
    df.index = df.index.map(lambda x: x[:-2])
    df['dif'] = df.doublet - df.singlet
    
    doublets = df[(df.prediction == 'doublet') & (df.dif > 0.5)]

    # =========================================================================
    # SECTION 3: RE-LOADING RAW DATA & REMOVING IDENTIFIED DOUBLETS
    # =========================================================================
    # Reload pristine un-subsumed raw matrix
    adata = sc.read_csv(csv_path).T
    
    # Ensure CSR sparse format for memory efficiency
    if not isinstance(adata.X, csr_matrix):
        adata.X = csr_matrix(adata.X)

    # 🔒 CRITICAL FIX: Lock pure raw integer counts into 'counts' layer NOW
    adata.layers["counts"] = adata.X.copy()

    sample_name = os.path.basename(csv_path).replace('_raw_counts.csv', '')
    adata.obs['Sample'] = sample_name
    
    # Remove predicted doublets
    adata.obs['doublet'] = adata.obs.index.isin(doublets.index)
    adata = adata[~adata.obs.doublet].copy()

    # =========================================================================
    # SECTION 4: CELL QUALITY CONTROL (QC) METRICS ANNOTATION
    # =========================================================================
    sc.pp.filter_cells(adata, min_genes=200)
    
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    
    ribo_url = "http://software.broadinstitute.org/gsea/msigdb/download_geneset.jsp?geneSetName=KEGG_RIBOSOME&fileType=txt"
    ribo_genes = pd.read_table(ribo_url, skiprows=2, header=None)
    adata.var['ribo'] = adata.var_names.isin(ribo_genes[0].values)
    
    # QC metrics on unnormalized integers
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt', 'ribo'], percent_top=None, log1p=False, inplace=True)

    # =========================================================================
    # SECTION 5: OUTLIER FILTERING & FINAL CLEANUP
    # =========================================================================
    upper_lim = np.quantile(adata.obs.n_genes_by_counts.values, 0.98)
    adata = adata[adata.obs.n_genes_by_counts < upper_lim].copy()
    adata = adata[adata.obs.pct_counts_mt < 15].copy()
    adata = adata[adata.obs.pct_counts_ribo < 20].copy()
    
    return adata

## NOTE: using .copy in section 5 of the function:
Adding .copy() forces Python to allocate new memory and create an independent AnnData object.

Why this is critical in Scanpy / scRNA-seq pipelines:
1. Prevents ImplicitModificationWarning
* If you modify a view (e.g., adding downstream normalization, principal components, or clustering labels to adata.obs or adata.X), Scanpy will raise a runtime warning or error because you are attempting to mutate a slice of another object.

2. Frees Memory (Garbage Collection)
* When you filter out hundreds or thousands of low-quality cells, the underlying expression matrices (adata.X, adata.layers) remain anchored to the initial full-size dataset in RAM as long as the object is just a view. Calling .copy() creates a smaller, clean matrix containing only the remaining cells, allowing Python’s garbage collector to free up memory from the filtered-out doublets and dead cells.

3. Prevents Downstream Data Corruption
* Modifying an AnnData view in-place can inadvertently mutate the pre-filtered original array in unpredictable ways. .copy() breaks the link to the raw pre-QC matrix so downstream integration, dimensional reduction, and scaling run safely.

In [ ]:
wd= %pwd
wd

In [ ]:
#Run this quick snippet to confirm that the samples exist in the directory 
import os
input_dir = f'{wd}/covid19_sc_data/GSE171524_RAW/'
print("Files in directory:")
for f in os.listdir(input_dir):
    print(f)

---------
## 4.2 IMPLEMENTING Batch Quality Control and Filtering
---------

In [ ]:
#IMPLEMENTING Batch Quality Control and Filtering¶
import gc
import os
import scanpy as sc

# Define working directory and correct subfolder path
wd = os.getcwd()
input_dir = os.path.join(wd, "covid19_sc_data", "GSE171524_RAW")
print("Input Directory: ", input_dir)

output_dir = os.path.join(wd, "covid19_sc_data", "processed")
os.makedirs(output_dir, exist_ok=True)
print("Output Directory:", output_dir)

# 1. Filter valid files
files_to_process = [
    f
    for f in os.listdir(input_dir)
    if not f.startswith("._") and f.lower().endswith(".csv.gz")
]
total_files = len(files_to_process)

print(f"\nFound {total_files} total files to process.\n")

# 2. Iterate with progress tracking and skip check
for idx, file in enumerate(files_to_process, start=1):
    csv_path = os.path.join(input_dir, file)

    # Clean sample identifier and define output file path
    sample_name = os.path.basename(csv_path).replace("_raw_counts.csv.gz", "")
    output_path = os.path.join(output_dir, f"{sample_name}.h5ad")

    # CHECK: Skip if destination .h5ad already exists
    if os.path.exists(output_path):
        print(
            f"Skipping ({idx}/{total_files}): {sample_name}.h5ad already exists."
        )
        continue

    print(f"Processing ({idx}/{total_files}): {file}...")

    # Execute preprocessing pipeline
    adata = pp(csv_path, n_epocs = 200)

    # Save to high-performance AnnData h5ad format
    adata.write(output_path)
    print(f"Saved ({idx}/{total_files}): {output_path}\n")

    # Explicit memory garbage collection
    del adata
    gc.collect()

print("🎉 All remaining files successfully processed and saved.")

-----------------------------------

### NOTE OLD IMPLEMENTATION OF THE FUNCTION
-----------------

"""


OLDER VERSION: 

out = []

for file in os. listdir ('raw_counts/'):

    out append (pp(' raw_counts/' + file)) 

    
""""

--------------

The previous version had two critical issues that break when processing single-cell datasets at scale: RAM saturation and lack of persistence.

Key Improvements:
1. Prevents Memory Crashes (OOM Errors)

* Old: out.append(...) kept every processed AnnData object inside a single Python list in RAM simultaneously. With single-cell matrices, loading dozens of samples into memory quickly causes Out-Of-Memory (OOM) crashes or heavy swap usage.

* New: Processing one file at a time, calling del adata, and triggering gc.collect() explicitly clears the memory buffer after each sample finishes.

2. Per-Sample Disk Persistence (.h5ad)

* Old: Held everything in active Python memory without saving to disk. If the script crashed on sample 15 out of 20, all progress was lost.

* New: Writes each processed sample to .h5ad (HDF5-backed AnnData format). If execution breaks midway, completed samples remain safely saved on disk.

3. File Filter Safety

* Old: Processed every item in raw_counts/, including hidden system files like .DS_Store or non-data files.

* New: if not file.lower().endswith('.csv.gz'): continue guarantees that only valid target files pass into the preprocessing function.




---------
## 4.3 CONCATENATION & METADATA MERGING
---------

In [ ]:
%pwd

In [ ]:
import gc
import os
import anndata as ad
import scanpy as sc
from scipy.sparse import csr_matrix

# 0. Setting up Dirs
wd = os.getcwd()
input_dir = os.path.join(wd, "covid19_sc_data", "processed")
print("Input Directory: ", input_dir)

# Define output DIRECTORY and ensure it exists
output_dir = os.path.join(wd, "covid19_sc_data", "processed", "concatenated")
os.makedirs(output_dir, exist_ok=True)
print("Output Directory:", output_dir)

# Define output FILE path
output_path = os.path.join(output_dir, "combined_concat.h5ad")

# 1. Obtener la lista de archivos limpios
files = sorted(
    [
        os.path.join(input_dir, f)
        for f in os.listdir(input_dir)
        if f.endswith(".h5ad") and not f.startswith("._")
    ]
)

print(f"Archivos a concatenar: {len(files)}")

# 2. Cargar cada muestra optimizando memoria RAM
adatas = {}
for f in files:
    # Extraer ID limpio de la muestra (ej. "GSM5226574_C51ctr")
    sample_id = os.path.basename(f).replace(".h5ad", "")

    # Cargar archivo
    ad_temp = sc.read_h5ad(f)

    # Convertir X a CSR Sparse Matrix (Reduce el uso de RAM un ~70%)
    if not isinstance(ad_temp.X, csr_matrix):
        ad_temp.X = csr_matrix(ad_temp.X)

    # Limpiar capas innecesarias que consumen RAM
    ad_temp.uns.clear()
    ad_temp.obsm.clear()

    # Asignar la columna Sample explícitamente
    ad_temp.obs["Sample"] = sample_id
    adatas[sample_id] = ad_temp

print("\nConcatenando todas las muestras en un solo paso...")

# 3. Concatenación masiva en un solo paso
adata_combined = ad.concat(
    adatas, label="Sample", index_unique="_", merge="unique"
)

# Liberar memoria de los objetos individuales
del adatas
gc.collect()

# 4. Filtrado global de genes
sc.pp.filter_genes(adata_combined, min_cells=150)

# 5. Limpieza de nombres de Sample
adata_combined.obs["Sample"] = (
    adata_combined.obs["Sample"]
    .astype(str)
    .str.split("_")
    .str[-1]
    .astype("category")
)

# 6. Guardar archivo maestro
print(f"Guardando archivo final en: {output_path}")
adata_combined.write_h5ad(output_path)

print("🎉 Concatenación exitosa y guardada en SSD sin problemas de memoria.")

In [ ]:
# Concatenation with Automatic Metadata Deduplication
import gc
import os
import anndata as ad
import scanpy as sc
from scipy.sparse import csr_matrix
# 0. Setting up Dirs
wd = os.getcwd()
input_dir = os.path.join(wd, "covid19_sc_data", "processed")
print("Input Directory: ", input_dir)

# Define output DIRECTORY and ensure it exists
output_dir = os.path.join(wd, "covid19_sc_data", "processed", "concatenated")
os.makedirs(output_dir, exist_ok=True)
print("Output Directory:", output_dir)

# Define output FILE path
output_path = os.path.join(output_dir, "combined_concat.h5ad")

# 1. Obtener la lista de archivos limpios
files = sorted(
    [
        os.path.join(input_dir, f)
        for f in os.listdir(input_dir)
        if f.endswith(".h5ad") and not f.startswith("._")
    ]
)

print(f"Archivos a concatenar: {len(files)}")

# 2. Cargar cada muestra optimizando memoria RAM
adatas = {}
for f in files:
    # Extraer ID limpio de la muestra (ej. "GSM5226574_C51ctr")
    sample_id = os.path.basename(f).replace(".h5ad", "")

    # Cargar archivo
    ad_temp = sc.read_h5ad(f)

    # Convertir X a CSR Sparse Matrix (Reduce el uso de RAM un ~70%)
    if not isinstance(ad_temp.X, csr_matrix):
        ad_temp.X = csr_matrix(ad_temp.X)

    # Limpiar capas innecesarias que consumen RAM
    ad_temp.uns.clear()
    ad_temp.obsm.clear()

    # Asignar la columna Sample explícitamente
    ad_temp.obs["Sample"] = sample_id
    adatas[sample_id] = ad_temp

print("\nConcatenando todas las muestras en un solo paso...")

# 3. Concatenación masiva en un solo paso
adata_combined = ad.concat(
    adatas, label="Sample", index_unique="_", merge="unique"
)

# Liberar memoria de los objetos individuales
del adatas
gc.collect()

# =========================================================================
# CRITICAL FIX: ELIMINATE DUPLICATED OBS COLUMNS CREATED BY AD.CONCAT
# =========================================================================
if adata_combined.obs.columns.duplicated().any():
    print("Found duplicate columns in .obs post-concatenation. Deduplicating...")
    adata_combined.obs = adata_combined.obs.loc[
        :, ~adata_combined.obs.columns.duplicated()
    ].copy()

# 4. Filtrado global de genes
sc.pp.filter_genes(adata_combined, min_cells=150)

# 5. Limpieza y categorización de la columna 'Sample'
adata_combined.obs["Sample"] = (
    adata_combined.obs["Sample"]
    .astype(str)
    .str.split("_")
    .str[-1]
    .astype("category")
)

# 6. Guardar archivo maestro
print(f"Guardando archivo final en: {output_path}")
adata_combined.write_h5ad(output_path)

print("🎉 Concatenación exitosa y guardada en SSD sin problemas de memoria.")

$$$$$$$$$$$$$$$$

-------- 
## NOTA SOBRE PORQUE EL CAMBIO EN ESTA FORMA DE CONCATENAR:
* Matriz Dispersa (csr_matrix): Las matrices densas de $110,000 \text{ células} \times 20,000 \text{ genes}$ consumen en RAM más de 16 GB. Al forzar la conversión a csr_matrix, solo almacena los valores distintos de cero, reduciendo el consumo a menos de 2 GB.
* Un solo paso (ad.concat): Evita hacer sc.concat dentro de un bucle for (el cual creaba copias duplicadas de la matriz combinada en cada iteración y mataba el kernel).
* Limpieza previa (uns y obsm): Remueve cualquier matriz intermedia o cálculo previo de las muestras individuales para mantener el objeto maestro ligero antes del entrenamiento con scVI.
$$$$$$$$$$$$

---------
## 4.3.1 CHECKING DATA INTEGRITY AFTER CONCATENATION AND MERGING
---------

In [ ]:
import pandas as pd
import os

# 0. Load the concatenated AnnData object 
wd = os.getcwd()
adata_path = os.path.join(wd, "covid19_sc_data", "processed", "concatenated",'combined_concat.h5ad')

print("Loading Concatenated ADATA from: ", adata_path)

# /Users/beto/progra25/covid19_single_cell_analysis/covid19_sc_data/processed/concatenated/combined_concat.h5ad

adata_combined = sc.read_h5ad(adata_path)

# 1. Inspect total cell counts per sample
print("Cell counts per sample:")
print(adata_combined.obs["Sample"].value_counts())

In [ ]:
# 2. Confirm unique sample count
total_samples = adata_combined.obs["Sample"].nunique()
print(f"\nTotal unique samples loaded: {total_samples} / Expected: 4")

In [ ]:
adata_combined.obs

In [ ]:
# 1. Limpiar y categorizar la columna 'Sample'
adata_combined.obs["Sample"] = (
    adata_combined.obs["Sample"]
    .astype(str)
    .str.split("_")
    .str[-1]
    .astype("category")
)

# 2. Corregir y acortar los nombres de las células (obs_names)
adata_combined.obs_names = [
    f"{bc.split('_')[0]}_{sample}"
    for bc, sample in zip(adata_combined.obs_names, adata_combined.obs["Sample"])
]

# 3. Verificar el resultado
adata_combined.obs
    

In [ ]:
adata_combined.obs.groupby('Sample').count()

-----
-----
-----
-----
-----
-----
-----
-----


# 5. BATCH PROCESSING AND MERGING (BEFORE INTEGRATION)


-----
-----
-----
-----
-----
-----
-----
-----

In [ ]:
import scanpy as sc
import scvi
import seaborn as sns
import numpy as np
import gc
import pandas as pd

---------
### 5.1 DATA PREP
---------

In [ ]:
#We convert the X matrix to a "sparce matrix" to reduce file size and memory
# from scipy.sparse import csr_matrix 
# adata.X = csr_matrix(adata.X)

## --> already done during concatenation

In [ ]:
# Save unnormalized raw counts for scVI
adata_combined.layers["counts"] = adata_combined.X.copy()

# Log-normalize main X matrix for standard plotting & marker analysis
sc.pp.normalize_total(adata_combined, target_sum=1e4)
sc.pp.log1p(adata_combined)
adata_combined.raw = adata_combined

##A. HERE Scanpy toakes a permanent snapshot of your total-count normalized and
## log-transformed matrix ($\log(1 + \text{CPM})$) and stored it in the .raw attribute.

## B. After that snapshot, even if you filter high-variable genes, run scVI, or modify adata.X, 
## the full set of normalized genes remains safely saved inside adata_combined.raw.

## C. When you run sc.pl.umap(adata_combined, color=['PTPRC', 'CD3E', 'CD4']), 
## Scanpy automatically checks if .raw exists. If it does, it pulls the gene expression directly from adata_combined.raw.

### D. Memory Efficiency: adata.raw keeps the data stored as a sparse matrix (csr_matrix). Sparse matrices only store non-zero values, consuming around 1–2 GB of RAM.

## E. scvi_normalized: Generates a dense matrix where every single zero is calculated as a non-zero float value (e.g., 0.0000412), expanding the memory footprint to 18+ GB of RAM.

# NOTE: we are gonna use 4 different annotations to correct our data
* Sample
* total counts
* mitochondrial counts
* ribosomal counts

In [ ]:
adata_combined.obs.head()

In [ ]:
adata_combined
#number of cells much higher than gene number (CELLS X GENES), so no need to filter, but if you needed: 


---------
### 5.2 TRAINNING MODEL: INTEGRATION & LATENT REPRESENTATION LEARNING (scVI)
---------

In [ ]:
import scvi

# 1. Setup AnnData (FIXED duplicate continuous covariate key)
scvi.model.SCVI.setup_anndata(
    adata_combined, 
    layer="counts", 
    categorical_covariate_keys=["Sample"],
    continuous_covariate_keys=["pct_counts_mt", "total_counts", "pct_counts_ribo"]  # Fixed duplicate key
)

# 2. Instantiate Model
model_combined = scvi.model.SCVI(adata_combined)

# 3. Train with reduced epochs and Early Stopping
model_combined.train(
    max_epochs=200,                # Reduced from default ~400
    early_stopping=True,           # Automatically stops if validation loss stops improving
    early_stopping_patience=15,    # Waits 15 epochs without improvement before stopping
    accelerator="mps"
)

# 4. Extract Latent Representation
adata_combined.obsm["X_scVI"] = model_combined.get_latent_representation()

print("🎉 Model trained and latent space extracted!")
# --------------------------------------------------------------------------------------------------
# 3. INSPECT MODEL PERFORMANCE & METRICS (PUT IT HERE)
# --------------------------------------------------------------------------------------------------
# View available metrics logged during training
print("Available training history keys:")
print(model_combined.history.keys())

# Example: Inspect the last 5 values of the training ELBO loss
print("\nRecent ELBO Train Loss:")
print(model_combined.history["elbo_train"].tail())

#--------------------------------------------------------------------------------------------------
# NOTA DE MEMORIA Y OPTIMIZACIÓN:
# NO ejecutamos: adata_combined.layers["scvi_normalized"] = model_combined.get_normalized_expression()
#
# ¿Por qué?
# 1. Denso & Pesado: `get_normalized_expression()` convierte la matriz dispersa (sparse) en una 
#    matriz densa de punto flotante de gen x célula, lo que requiere más de 17 GB de memoria RAM.
# 2. Tamaño en disco: Si la guardamos como 'layer', el archivo .h5ad final pesará más de 20 GB.
# 3. Innecesario para Clustering: Los algoritmos de reducción de dimensionalidad (UMAP/Leiden) 
#    NO utilizan esta capa. Trabajan directamente sobre la representación latente integrada.
#
# CONCLUSIÓN:
# Para el pipeline de integración y clustering nos enfocaremos en:
#   1. `adata_combined.obsm["X_scVI"]`  --> Espacio latente integrado para UMAP y vecinos (Neighbors).
#   2. `adata_combined.X` (o counts)   --> Para el análisis de expresión diferencial posterior.
# --------------------------------------------------------------------------------------------------

In [ ]:
import matplotlib.pyplot as plt

# 1. Safely extract history metrics
history = model_combined.history

# 2. Check available keys to be sure
print("Available keys in history:", list(history.keys()))

plt.figure(figsize=(8, 5))

# Plot training ELBO (Check for common key names)
if "elbo_train" in history:
    plt.plot(history["elbo_train"], label="Train ELBO Loss", color="#1f77b4", linewidth=2)
elif "elbo_train_set" in history:
    plt.plot(history["elbo_train_set"], label="Train ELBO Loss", color="#1f77b4", linewidth=2)

# Safely plot validation ELBO if present
if "elbo_validation" in history:
    plt.plot(history["elbo_validation"], label="Validation ELBO Loss", color="#ff7f0e", linewidth=2)
elif "elbo_validation_set" in history:
    plt.plot(history["elbo_validation_set"], label="Validation ELBO Loss", color="#ff7f0e", linewidth=2)

plt.title("scVI Training History (ELBO Loss)")
plt.xlabel("Epochs")
plt.ylabel("ELBO Loss")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 1. Key Takeaways from the Plot

* Smooth Drop: The curve drops sharply within the first 20 epochs (from ~3150 down to ~2850), which means the model rapidly learned the primary structure of your scRNA-seq expression data.
* Plateau / Convergence: Around epochs 120–140, the slope flattens out almost completely near ~2795. This indicates that the variational autoencoder has reached convergence and additional epochs wouldn't yield significant improvements.
* No Oscillations: The curve is extremely smooth, confirming that the learning rate and batch sizes were well-suited for training on your Apple Silicon setup.


$$$$$$$$$$$$$$$$
# --> %%%%%%%%%%%%%%  CHECKPOINT !!!
$$$$$$$$$$$$$$$$

## 5.3 SAVING MODEL AND ANDATA

In [ ]:
import os
import gc
import scanpy as sc
import scvi

# -----------------------------------------------------------------------------
# SAVE CHECKPOINT
# -----------------------------------------------------------------------------
wd = os.getcwd()

save_dir = os.path.join(wd, "covid19_sc_data", "processed", "integrated_data")
os.makedirs(save_dir, exist_ok=True)

# 1. Save integrated AnnData
adata_output_path = os.path.join(save_dir, "integrated_adata.h5ad")
print(f"Saving integrated AnnData to: {adata_output_path}")
adata_combined.write_h5ad(adata_output_path)

# 2. Save scVI model
model_dir = os.path.join(wd, "covid19_sc_data", "saved_models", "model_combined")
os.makedirs(model_dir, exist_ok=True)

print(f"Saving scVI model to: {model_dir}")
model_combined.save(model_dir, overwrite=True)

gc.collect()
print("🎉 Integrated AnnData and scVI model successfully saved locally!")

## NOTE ON 5.2 TRAINNING MODEL¶

1. etup_anndata(...): Le indica a scVI dónde están los conteos crudos intactos (layer="counts") y cuál es la columna que genera el ruido técnico (categorical_covariate_keys=["Sample"]).

2. model_combined.train(accelerator="mps"): Entrena una red neuronal (un Variational Autoencoder o VAE) en la GPU de tu Mac M1 (mps). La red aprende a separar la biología real del ruido técnico.

get_latent_representation(): Comprime los ~20,000 genes de cada célula a solo 30 dimensiones biológicas limpias y las guarda en adata_combined.obsm["X_scVI"]. Esto es lo que usaremos para construir el UMAP y hacer el clustering.

---------
# =========================================================================
# 5.3 GRAPH NEIGHBORS, UMAP & LEIDEN CLUSTERING
# =========================================================================
---------

In [ ]:
### LOADING CHECKPOINT!


import pandas as pd
import numpy as np
import seaborn as sns
import gc
import os
import shutil
import scanpy as sc
import scvi

# -----------------------------------------------------------------------------
# LOADING CHECKPOINT
# -----------------------------------------------------------------------------
print("\n--- LOADING CHECKPOINT ---")

wd = os.getcwd()
adata_path = os.path.join(wd, "covid19_sc_data", "processed", "integrated_data", "integrated_adata.h5ad")
model_path = os.path.join(wd, "covid19_sc_data", "saved_models", "model_combined")

# 1. Load AnnData
adata_combined = sc.read_h5ad(adata_path)

# Ensure unique observation index strings
if not adata_combined.obs_names.is_unique:
    adata_combined.obs_names_make_unique()

# 2. Load scVI Model & Assign MPS Device
model_combined = scvi.model.SCVI.load(model_path, adata=adata_combined)
model_combined.to_device("mps")

print("🎉 Successfully loaded AnnData and scVI model on MPS acceleration!")

# --------------------------------------------------------------------------------------------------

## CALCULATING THE NEIGHBOURS

# --------------------------------------------------------------------------------------------------

* sc.pp.neighbors(adata, use_rep = 'X_scVI')
* print("Calculating neighbor graph using 'X_scVI'...")
* sc.pp.neighbors(adata_combined, use_rep="X_scVI") #n_neighbors=15

*  Neighbor Graph: sc.pp.neighbors(...) calculated cell-to-cell distances using 
* X_scVI and saved them into adata_combined.uns['neighbors'] and adata_combined.obsp.

In [ ]:
%pip install leidenalg

### NOTE ON THESE PREVIOUs COMMANDS
* UMAP Coordinates: sc.tl.umap(...) computed the 2D coordinates and saved them in adata_combined.obsm['X_umap'].
* Leiden Cluster Assignments: sc.tl.leiden(...) assigned every single cell a cluster ID (e.g. cluster 0 to 22) and saved it in adata_combined.obs['leiden'].

### NOTE:
IT can be noted that in the second umap all the clusters have colors from all the different
samples which speaks of a good integration with the exception of 1 cluster belongin to
only one sample which could be a specific cell type different from the other samples
acquired during collection.

---------
## 5.2 Differential Expression Analysis & Marker Gene Discovery
---------

This section computes cluster-specific marker genes using two complementary statistical approaches:
1. **Scanpy (Frequentist / Wilcoxon Rank-Sum):** Fast ranking on log-normalized counts.
2. **scVI (Bayesian Generative Sampling):** Denoised differential expression directly from the VAE latent space.

---

In [ ]:
import scanpy as sc
import scvi

# =========================================================================
# STEP 1: FIX NEIGHBORS & LEIDEN ON INTEGRATED LATENT SPACE
# =========================================================================

# Ensure unique observation names
adata_combined.obs_names_make_unique()

# 1. Compute KNN graph using scVI latent embeddings
print("Computing neighbor graph on X_scVI...")
sc.pp.neighbors(adata_combined, use_rep="X_scVI", n_neighbors=20)

# 2. Compute 2D UMAP embedding
print("Computing UMAP...")
sc.tl.umap(adata_combined, init_pos="pca")
# this computes the 2D coordinates and saves them in adata_combined.obsm['X_umap'].

# 3. Run Leiden Clustering
res = 0.5
print(f"Running Leiden clustering (resolution={res}...")
sc.tl.leiden(adata_combined, resolution=res, key_added="leiden") # COME BACK AND CHANGE HERE IF YOU NEED TO SOLVE CLUSTERS below...
# this command assigns every single cell a cluster ID (e.g. cluster 0 to 22) and saved it in adata_combined.obs['leiden'].


# 4. Cast cluster labels to categorical
adata_combined.obs["leiden"] = adata_combined.obs["leiden"].astype("category")

print(f"Total unique clusters found: {adata_combined.obs['leiden'].nunique()}")



In [ ]:
# =========================================================================
# STEP 2: PLOT CLEAN UMAPS
# =========================================================================
import matplotlib.pyplot as plt
import scanpy as sc

# Create side-by-side axes
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Subplot 1: Sample (Legend on the right, no text on data)
sc.pl.umap(
    adata_combined,
    color="Sample",
    frameon=False,
    ax=axes[0],
    show=False
)

# Subplot 2: Leiden (Cluster IDs on top of clusters)
sc.pl.umap(
    adata_combined,
    color="leiden",
    frameon=False,
    legend_loc="on data",
    legend_fontsize=10,
    ax=axes[1],
    show=False
)

plt.tight_layout()
plt.show()

### CHECKPOINT UMAP ADATA

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import gc
import os
import shutil
import scanpy as sc
import scvi


# Save the AnnData object
wd = os.getcwd()
save_dir = os.path.join(wd, "covid19_sc_data", "processed", "umap_adata")
os.makedirs(save_dir, exist_ok=True)
adata_output_path = os.path.join(save_dir, "umap_adata.h5ad")

print(f"Saving umap AnnData to: {adata_output_path}")
adata_combined.write_h5ad(adata_output_path)
print("🎉 Successfully saved AnnData without duplicate column errors!")

# FINDING MARKERS

#### * LOADING UMAP ADATA

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import gc
import os
import shutil
import scanpy as sc
import scvi

# Setup paths
wd = os.getcwd()
adata_path = os.path.join(wd, "covid19_sc_data", "processed",  "umap_adata", "umap_adata.h5ad")
model_path = os.path.join(wd, "covid19_sc_data", "saved_models", "model_combined")


# 1. Load AnnData
umap_adata= sc.read_h5ad(adata_path)

# Ensure unique observation index strings
if not adata_combined.obs_names.is_unique:
    adata_combined.obs_names_make_unique()

# 2. Load scVI Model & Assign MPS Device
model_combined = scvi.model.SCVI.load(model_path, adata=umap_adata)
model_combined.to_device("mps")

print("🎉 Successfully loaded AnnData and scVI model on MPS acceleration!")

In [ ]:
# stores the differential expression statistics in adata_combined.uns['rank_genes_groups'].
# That is why plotting functions like sc.pl.rank_genes_groups() and sc.pl.rank_genes_groups_dotplot() work immediately after running it!
sc.tl.rank_genes_groups(umap_adata, 'leiden') #this updates layers in the adata object that we can plot to check "elbows"
## THIS COMMAND: stores cluster statistics inside adata_combined.uns['rank_genes_groups'].


In [ ]:
# Then thanks, show top 20 genes for each cluster
sc.pl.rank_genes_groups(umap_adata, n_genes=20, sharey=False)


In [ ]:
# Dotplot shows both expression level (color) and % of cells expressing the gene (dot size)
sc.pl.rank_genes_groups_dotplot(umap_adata, n_genes=3, values_to_plot="logfoldchanges")

In [ ]:
# 1A. Extract Wilcoxon rank-sum markers from Scanpy's rank_genes_groups
markers_scanpy = sc.get.rank_genes_groups_df(umap_adata, None)
markers_scanpy 

# get dataframe where the group is the laden cluster

In [ ]:
markers_scanpy.columns

In [ ]:
# Filter for significant up-regulated markers (FDR < 0.05 and logFC > 0.5)
markers_filtered = markers_scanpy.loc[
    (markers_scanpy.pvals_adj < 0.05) & (markers_scanpy.logfoldchanges > 0.5)
].copy()

markers_filtered

---------
### 5.3 scvi model and filtering
---------

In [ ]:
#Extract scVI Bayesian differential expression markers (Recommended)
markers_scvi   = model_combined.differential_expression(
    groupby="leiden", fdr_target=0.05
)

In [ ]:
print(markers_scvi.columns)

In [ ]:
import os
import numpy as np

# -----------------------------------------------------------------------------
# 1. COMPUTE METRICS & FILTER scVI MARKERS
# -----------------------------------------------------------------------------
# Compute log2 fold change from scVI normalized means (+1e-6 to avoid log(0))
markers_scvi["lfc_computed"] = np.log2(
    (markers_scvi["raw_normalized_mean1"] + 1e-6)
    / (markers_scvi["raw_normalized_mean2"] + 1e-6)
)

# Filter for high probability (proba_m1 > 0.8) and positive upregulation (lfc > 0.5)
markers_scvi_filtered = markers_scvi.loc[
    (markers_scvi["proba_m1"] > 0.8) & (markers_scvi["lfc_computed"] > 0.5)
].copy()

# Sort by cluster and probability/lfc
markers_scvi_filtered = markers_scvi_filtered.sort_values(
    by=["group1", "proba_m1", "lfc_computed"], ascending=[True, False, False]
)

# Inspect top 10 markers for cluster 0
print("Top 10 markers for cluster 0:")
print(markers_scvi_filtered[markers_scvi_filtered["group1"] == "0"].head(10))



Where Do the Markers Come From?

* **Both Scanpy and scVI scan your entire dataset to find the genes that are strongly turned ON in one cluster and turned OFF (or low) in the others.**

* Once you get that list of top genes for a cluster, you look up what those specific genes do in biology. For example:

* * If Cluster 0's top turned-on genes are CD3D, CD4, and IL7R $\rightarrow$ You label Cluster 0 as CD4+ T Cells.
  * If Cluster 1's top turned-on genes are CD14, LYZ, and S100A9 $\rightarrow$ You label Cluster 1 as CD14+ Monocytes.
  * If Cluster 2's top turned-on genes are PPBP and PF4 $\rightarrow$ You label Cluster 2 as Platelets / Megakaryocytes.

Neither Scanpy nor scVI uses an external "curated database" or predefined library of marker genes. Instead, both methods compare gene expression in each cluster against all other clusters (1 vs. Rest) directly from your dataset, but they use fundamentally different math:

* **Scanpy (sc.tl.rank_genes_groups)** — Classical Frequentist Hypothesis TestingHow it works: It uses classical non-parametric statistical tests (default is the Wilcoxon rank-sum test) on your log-normalized counts.Mechanics: For Cluster 0, it takes the expression values of Gene $A$ in Cluster 0 cells and compares them against the expression values of Gene $A$ across all remaining cells (Clusters 1, 2, 3...). It computes a $p$-value and log fold change (logfoldchanges).Limitation: Frequentist testing assumes observation independence and can suffer from p-value inflation on single-cell datasets due to high cell counts and technical dropouts/noise.

* **scVI (model.differential_expression)** — Deep Generative Bayesian SamplingHow it works: It uses the deep neural network parameters learned by scVI to sample from the underlying denoised, batch-corrected expression distribution (the latent posterior) of each cell.Mechanics: For Cluster 0 vs. Rest, scVI samples thousands of latent expression vectors for cells in Cluster 0 (raw_normalized_mean1) and compares them against sampled vectors from all other clusters (raw_normalized_mean2).Posterior Probability (proba_m1): Instead of $p$-values, scVI calculates the Bayesian posterior probability that a gene is genuinely upregulated in Cluster 0 beyond a biological threshold ($\delta$). A value of proba_m1 = 0.95 means there is a 95% probability that the gene is higher in Cluster 0 after removing batch effects and dropout noise.

In [ ]:
#.pl.umap(adata_combined, color = ['Sample'], frameon=False)

In [ ]:
#sc.pl.umap(adata_combined, color = ['leiden'], frameon=False, legend_loc='on data')

---------
# 6 MANUAL LABELING
---------

* EASIEST TO START WITH CELL TYPE YOU KNOW TWILL BE THERE... LIKE IT DEPENDS ON THE TISSUE BUT FOR EXAMPLE. BLOOD CELLS OR IMMUNE CELLS ARE ALMOST ALWAYS PRESENT


### ** Cell marker resoruces **
* panglaodb.se
  

### NOTE
Manual labeling of clusters can be more accurate specally when you dont have a good
data set to map it to It is easier to start with cell types that will probably be there. If there
are blood cells you can increase resolution of clustering

### NOTE: 
If we need separate clusters to be more precise at identifying cells we can increase
resolution we can go back tho this line of code: sc.tl.leiden(adata, resolution = 0.5)

--------

In [ ]:
import scanpy as sc
import scvi
import seaborn as sns
import numpy as np
import gc
import pandas as pd




$$$$

---------
### 6.2 PLOTTING FUNCION
---------

$$$$

In [ ]:
#6.2 Plotting Function
import gc
import scanpy as sc


def plot_scvi_markers(
    genes,
    model,
    adata,
    ncols=3,
    cmap="magma",
    size=10,
    frameon=False,
    legend_loc="on data",
):
    """Plots scVI denoised gene expression on UMAP on-the-fly without storing dense layers.

    Parameters:
    -----------
    genes : list of str
        List of gene symbols to visualize (e.g., ['CD3D', 'CD4', 'CD8A'])
    model : scvi.model.SCVI
        Your trained scVI model object
    adata : AnnData
        Your primary AnnData object containing 'X_umap'
    ncols : int
        Number of grid columns for plotting
    cmap : str
        Colormap palette (e.g., 'magma', 'viridis', 'plasma')
    size : float
        Point size for UMAP scatter
    frameon : bool
        Whether to show plot frame border
    legend_loc : str
        Location of legend or colorbar
    """
    # 1. Ensure genes exist in the dataset
    available_genes = adata.var_names.union(
        adata.raw.var_names if adata.raw is not None else []
    )
    valid_genes = [g for g in genes if g in available_genes]

    if not valid_genes:
        print("❌ None of the requested genes were found in the dataset.")
        return

    # 2. Extract denoised normalized expression directly from scVI
    denoised_df = model.get_normalized_expression(gene_list=valid_genes)

    # 3. Store temporary columns in .obs
    plot_cols = []
    for g in valid_genes:
        col_name = f"{g}_denoised"
        adata.obs[col_name] = denoised_df[g].values
        plot_cols.append(col_name)

    # 4. Render clean UMAP with custom titles matching gene names
    titles = [g for g in valid_genes]

    sc.pl.umap(
        adata,
        color=plot_cols,
        title=titles,
        frameon=frameon,
        ncols=ncols,
        cmap=cmap,
        sort_order=True,  # FIXED: Changed from sort_value to sort_order
        size=size,
        legend_loc=legend_loc,
        vmax = vmax
    )

    # 5. Clean up temporary .obs columns to keep AnnData lightweight
    for col in plot_cols:
        del adata.obs[col]

    gc.collect()

In [ ]:
## DOTPLOT FUNCTION
import gc
import scanpy as sc


def plot_scvi_dotplot(
    genes,
    model,
    adata,
    groupby="leiden",
    standard_scale="var",
    cmap="Reds",
    title=None,
):
    """Plots a dotplot of scVI-denoised gene expression across clusters.

    Parameters:
    -----------
    genes : list of str
        List of gene symbols to visualize (e.g., ['CD4', 'CD8A', 'CD14'])
    model : scvi.model.SCVI
        Your trained scVI model object
    adata : AnnData
        Your primary AnnData object
    groupby : str
        Column in `adata.obs` to group cells by (default: 'leiden')
    standard_scale : {'var', 'group', None}
        Scales expression between 0 and 1 across genes ('var') for easy comparison
    cmap : str
        Colormap palette for expression intensity
    title : str
        Optional plot title
    """
    # 1. Ensure genes exist in the dataset
    available_genes = adata.var_names.union(
        adata.raw.var_names if adata.raw is not None else []
    )
    valid_genes = [g for g in genes if g in available_genes]

    if not valid_genes:
        print("❌ None of the requested genes were found in the dataset.")
        return

    # 2. Extract denoised expression directly from scVI
    denoised_df = model.get_normalized_expression(gene_list=valid_genes)

    # 3. Store temporary columns in .obs
    plot_cols = []
    for g in valid_genes:
        col_name = f"{g}_denoised"
        adata.obs[col_name] = denoised_df[g].values
        plot_cols.append(col_name)

    # 4. Map back to clean gene names for plotting labels
    var_names_dict = {col: g for col, g in zip(plot_cols, valid_genes)}

    # 5. Render DotPlot (dendrogram disabled to avoid resolution mismatch errors)
    sc.pl.dotplot(
        adata,
        var_names=plot_cols,
        groupby=groupby,
        standard_scale=standard_scale,
        cmap=cmap,
        dendrogram=False,
        var_group_rotation=45,
        title=title if title else f"scVI Denoised Expression by {groupby}",
    )

    # 6. Clean up temporary .obs columns
    for col in plot_cols:
        del adata.obs[col]

    gc.collect()

---------
# 6.3 SETUP FOR MANUAL LABELING
---------

In [ ]:
#DICTIONARY TEMPLATE FOR MANUAL LABELING:
for x in range (0,20):
    print(f"'{x}':'',")

## 6.3.1. High-Level Annotation Strategy
A top-down, hierarchical approach prevents confusion caused by overlapping subtype markers
* Compartment Level: Divide the total dataset into Epithelial, Endothelial, Stromal, and Immune (CD45+) compartments.   Lineage Level: Split the Immune compartment into Myeloid ($CD11b^+$) and Lymphoid ($CD3^+$ / $CD19^+$) branches.   Subtype Level: Assign fine cell types using lineage-specific marker panels.   ⚠️ Note on CD45 (PTPRC): Do not rely solely on PTPRC. Due to gene dropout in scRNA-seq, use a pan-immune panel (PTPRC, LAPTM5, SRGN, CORO1A) to capture all immune populations.   

$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$

## 6.3.2. Anatomical Super-Clusters (Hypothesis Based on UMAP Topology)
Based on UMAP spatial organization, the clusters group into 5 main anatomical islands to be verified with marker expression:

* Myeloid Super-Cluster: Likely tissue macrophages, monocytes, and dendritic cells driving pulmonary inflammation.
* Lymphoid Super-Cluster: Likely T cells ($CD4^+$, $CD8^+$) and NK cells.
* Epithelial Super-Cluster: Likely pneumocytes (AT1, AT2) and airway lining (ciliated, club cells).
* Stromal & Endothelial: Likely vascular and lymphatic vessels, structural fibroblasts, and pericytes.
* Isolated Lineages: Likely B/Plasma cells (Cluster 5) and Mast cells/Neutrophils


$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$

----------------
## 6.3.3. Comprehensive Reference Marker Panel
Use this dictionary with plot_scvi_dotplot() to test and evaluate marker expression across all clusters:

----------------

$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$

In [ ]:
#3. Comprehensive Reference Marker Panel
#Use this dictionary with plot_scvi_dotplot() to test and evaluate marker expression across all clusters:
#%load_ext autoreload
%reload_ext autoreload
%autoreload 2

from markers import COVID_LUNG_MARKERS

# Alias it so your existing code works without changing a word
covid_lung_markers = COVID_LUNG_MARKERS

for category, genes in covid_lung_markers.items():
    print(f"🔹 {category:<25} ({len(genes)} genes): {', '.join(genes)}")

#import json
#print(json.dumps(covid_lung_markers, indent=4))

| Cluster Group | Candidate Lineages to Test | Key Discriminating Markers to Check |
| :--- | :--- | :--- |
| **`?`** | Alveolar vs. Inflammatory Macrophages | Check `FABP4` / `MARCO` vs. `CD14` / `S100A8` |
| **`?`, `?`** | AT2 vs. Ciliated Epithelial Cells | Check `SFTPC` (AT2) vs. `FOXJ1` (Ciliated) |
| **`?`** | CD4+ vs. CD8+ T Cells | Check `CD3D` with `CD4` vs. `CD8A` |
| **`?`, `?`** | Vascular Endothelium vs. Fibroblasts | Check `PECAM1` (Endothelial) vs. `COL1A1` (Fibroblast) |
| **`?`** | B Cells vs. Plasma Cells | Check `MS4A1` (B) vs. `SDC1` / `MZB1` (Plasma) |
| **`?`, `?`** | Smooth Muscle vs. Pericytes | Check `ACTA2`, `RERGL` |
| **`?`, `?`** | NK Cells vs. Cytotoxic CD8+ T Cells | Check `NKG7` / `GNLY` without `CD3E` (NK) vs. `CD3E`+ (CD8+ T) |
| **`?`, `?`, `?`** | Classical Monocytes vs. Interstitial Macs vs. cDCs | Check `CD14` vs. `CD163` vs. `CD1C` |
| **`?`, `?`** | Secretory/Club vs. AT1 Pneumocytes | Check `SCGB1A1` (Club) vs. `AGER` (AT1) |
| **`?`** | Lymphatic Endothelial Cells | Check `PROX1`, `LYVE1` |
| **`?`, `?`** | Non-Classical Monocytes vs. pDCs | Check `FCGR3A` (CD16) vs. `LILRA4` (pDC) |
| **`?`** | Mast Cells vs. Granulocytes/Neutrophils | Check `TPSAB1` (Mast) vs. `CSF3R` (Neutrophil) |
| **`?`, `?`** | Basal Epithelial vs. Rare Immune Subtypes | Check top DE genes in `markers_scvi_filtered` |

In [ ]:
#to double check with gene marker
markers_filtered[markers_filtered.names == 'MT-CO1'] 

### NOTE: INCREASE RESOLUTION FOR UNRESOLVED CLUSTERS

If a clsuter is not clear and needs subdivision, we can increase resolution search above:
"""
#CREATING AD COLORING UMAP:
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution = 0.7) 

 
### HERE YOU CHANGE THE RESOLUTION to resolve clusters in 6. labeling !!!

""" 

THEN RERUN from: "5.2 FIND MARKERS FOR THE CLUSTERS"



In [ ]:
#Checking current clusters 
sc.pl.umap(umap_adata, color=['leiden'], frameon = False, legend_loc = 'on data')

In [ ]:
sc.pl.umap(umap_adata, color=['CD4', 'CD8A', 'PTPRC'], frameon=False, legend_loc = "on_data", vmax=3)

In [ ]:
sc.pl.umap(umap_adata, color=['CD4', 'CD8A', 'PTPRC'], frameon=False, legend_loc = "on_data", vmax=10)

In [ ]:
# #BEST PLOTTING WITH FUNCTION
# Plot key cell markers on-the-fly
marker_genes = ["CD4", "CD8A", "PTPRC", "CD14", "ITGAX", 'MRC1', 'MARCO',]

plot_scvi_markers(
    genes=marker_genes,
    model=model_combined,
    adata=umap_adata,
    ncols=3,
    #cmap="magma",
    size=8,
)

plot_scvi_dotplot(
    genes=marker_genes,
    model=model_combined,
    adata=umap_adata,
    groupby="leiden",  # Uses active resolution (e.g., 0.9)
    standard_scale="var",  # Normalizes each gene from 0 to 1 so high/low expressors are easily comparable
)

In [ ]:
#to double check with group number
markers_filtered[markers_filtered.group == '10'].head(10)

In [ ]:

markers_scanpy_filtered[markers_scanpy_filtered.names == 'CD4']

$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$

--------------------
# 6.4 RECLUSTERING
---------
Some clusters are not resolved according to the markers tried above (cluster 7 for example), so increasing resolution would be a good idea...

We can load the exact same 'model_combined'  saved right after training.

Here is the key distinction to prevent confusion:
* scvi.model.SCVI model:  We never need to save a separate model for UMAP/clustering (there is no such thing as model_umap). The scVI model's only job is to learn the latent space $Z$ (X_scVI) and model gene expression distributions. It doesn't care about Leiden clusters or UMAP coordinates.
* umap_adata (AnnData): This is where all downstream, interactive results live—including X_scVI, neighbors, X_umap, and obs['leiden'].When re-running differential expression with scVI at a new resolution, scVI simply queries the trained model_combined using the cell groupings stored inside umap_adata.obs['leiden']

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import scvi

# Set plotting defaults
sc.settings.verbosity = 2
sc.set_figure_params(dpi=120, frameon=False)


# =========================================================================
# STEP 1: INITIALIZE PATHS & LOAD CHECKPOINT
# =========================================================================
wd = os.getcwd()

adata_path = os.path.join(wd, "covid19_sc_data", "processed",  "umap_adata", "umap_adata.h5ad")
model_path = os.path.join(wd, "covid19_sc_data", "saved_models", "model_combined")

print("1. Loading AnnData with precomputed UMAP and neighbor graph...")
umap_adata = sc.read_h5ad(adata_path)

if not umap_adata.obs_names.is_unique:
    umap_adata.obs_names_make_unique()

print("2. Loading trained scVI model...")
model_combined = scvi.model.SCVI.load(model_path, adata=umap_adata)
model_combined.to_device("mps")  # Change to "cuda" or "cpu" if needed

print("✓ Checkpoint loaded successfully!\n")

In [ ]:
# =========================================================================
# STEP 2: INTERACTIVE RESOLUTION EXPLORER (RUN & RE-RUN THIS BLOCK)
# =========================================================================
test_res = 0.9  # Set to whatever resolution you want to test (e.g., 0.5, 0.7, 0.9)

col_name = f"leiden_res_{str(test_res).replace('.', '_')}"

print(f"--- Testing Leiden resolution = {test_res} ---")

# 1. Compute Leiden clustering on existing graph (Instant)
sc.tl.leiden(umap_adata, resolution=test_res, key_added=col_name)

# 2. Update active 'leiden' key
umap_adata.obs["leiden"] = umap_adata.obs[col_name].astype("category")

# 3. Clear stale DE / dendrogram computations from prior resolutions
umap_adata.uns.pop("rank_genes_groups", None)
umap_adata.uns.pop("dendrogram_leiden", None)

print(f"Total clusters identified: {umap_adata.obs['leiden'].nunique()}")

# 4. Plot UMAP
sc.pl.umap(
    umap_adata,
    color=["leiden"],
    legend_loc="on data",
    legend_fontsize=10,
    title=f"Leiden Clustering (Resolution {test_res})"
)

# 5. Quick Scanpy Marker Check
print("Computing fast Wilcoxon rank-sum markers for inspection...")
sc.tl.rank_genes_groups(umap_adata, groupby="leiden", method="wilcoxon")

# 6. Plot top 3 markers per cluster
sc.pl.rank_genes_groups_dotplot(
    umap_adata,
    n_genes=3,
    values_to_plot="logfoldchanges",
    min_logfoldchange=0.5,
    dendrogram=False,  # Prevents dendrogram mismatch error across resolutions
    title=f"Top Wilcoxon Markers (Res {test_res})"
)

In [ ]:
# =========================================================================
# STEP 3: LOCK IN FINAL RESOLUTION & SAVE CHECKPOINT (RUN ONCE DONE)
# =========================================================================
chosen_res = test_res
print(f"\n==================================================")
print(f"Locking in final resolution: {chosen_res}")
print(f"==================================================")

# 1. Compute full scVI Bayesian Differential Expression
print("\nRunning scVI Bayesian DE on final clusters (this queries model_combined)...")
markers_scvi = model_combined.differential_expression(
    groupby="leiden",
    fdr_target=0.05
)

# Calculate log2 fold changes from scVI normalized expression
markers_scvi["lfc_computed"] = np.log2(
    (markers_scvi["raw_normalized_mean1"] + 1e-6)
    / (markers_scvi["raw_normalized_mean2"] + 1e-6)
)

# Filter for confident scVI marker genes
markers_scvi_filtered = markers_scvi.loc[
    (markers_scvi["proba_m1"] > 0.8) & (markers_scvi["lfc_computed"] > 0.5)
].copy().sort_values(
    by=["group1", "proba_m1", "lfc_computed"],
    ascending=[True, False, False]
)

# 2. Export CSV files & Update AnnData on Disk
res_str = str(chosen_res).replace('.', '_')
save_dir = os.path.join(processed_dir, "markers_and_clustering")
os.makedirs(save_dir, exist_ok=True)

# Save filtered scVI DE table
scvi_csv_path = os.path.join(save_dir, f"scvi_markers_leiden_{res_str}.csv")
markers_scvi_filtered.to_csv(scvi_csv_path)

# Extract and save Scanpy DE table
markers_scanpy = sc.get.rank_genes_groups_df(umap_adata, group=None)
markers_scanpy_filtered = markers_scanpy.loc[
    (markers_scanpy.pvals_adj < 0.05) & (markers_scanpy.logfoldchanges > 0.5)
].copy()
scanpy_csv_path = os.path.join(save_dir, f"scanpy_markers_leiden_{res_str}.csv")
markers_scanpy_filtered.to_csv(scanpy_csv_path)

# Save final updated AnnData object (contains all resolution columns tested in .obs)
umap_adata.write_h5ad(adata_path)

print(f"\n🎉 PIPELINE COMPLETE & SAVED TO DISK!")
print(f" - Updated AnnData saved to: {adata_path}")
print(f" - scVI Marker CSV saved to: {scvi_csv_path}")
print(f" - Scanpy Marker CSV saved to: {scanpy_csv_path}")

In [ ]:
# =========================================================================
# STEP 4: LOAD CHECKPOINT TO KEEP ANALYZING
# =========================================================================
wd = os.getcwd()
chosen_res = 
res_str = str(chosen_res).replace('.', '_')
adata_path = os.path.join(wd, "covid19_sc_data", "processed",  "umap_adata", "umap_adata.h5ad")
model_path = os.path.join(wd, "covid19_sc_data", "saved_models", "model_combined")
markers_path = os.path.join(wd, "covid19_sc_data",  "markers_and_clustering",  f"scanpy_markers_leiden_{res_str}.csv")




$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$

---------
# 6.5 MARKER HUNTING 
---------

$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$

In [ ]:
#3. Comprehensive Reference Marker Panel
#Use this dictionary with plot_scvi_dotplot() to test and evaluate marker expression across all clusters:
#%load_ext autoreload
%reload_ext autoreload
%autoreload 2

from markers import COVID_LUNG_MARKERS

# Alias it so your existing code works without changing a word
covid_lung_markers = COVID_LUNG_MARKERS

for category, genes in covid_lung_markers.items():
    print(f"🔹 {category:<25} ({len(genes)} genes): {', '.join(genes)}")

#import json
#print(json.dumps(covid_lung_markers, indent=4))

$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$

# 6.5.1 GLBOAL MARKERS

In [ ]:
sc.pl.umap(
    umap_adata,
    color=["leiden"],
    legend_loc="on data",
    legend_fontsize=10,
    title=f"Leiden Clustering (Resolution {test_res})"
)


In [ ]:
import json

# 1. Select ONLY the general broad categories
broad_categories = [
    "Pan_Immune",
    "General_Myeloid",
    "General_Lymphoid",
    "General_Epithelial",
    "General_Stromal",
    "General_Endothelial"
]

sub_markers = {cat: covid_lung_markers[cat] for cat in broad_categories if cat in covid_lung_markers}

print("Testing General Compartments:")
print(json.dumps(sub_markers, indent=4))

genes_to_plot = [gene for sublist in sub_markers.values() for gene in sublist]

# 2. Run Dotplot across all 22 clusters

plot_scvi_markers(
    genes=genes_to_plot,
    model=model_combined,
    adata=umap_adata,
    ncols=3,
    cmap="magma",
    size=8,
    
)



plot_scvi_dotplot(
    genes=genes_to_plot,
    model=model_combined,
    adata=umap_adata,
    groupby="leiden",
)

$$$$
# 🔬 Global Markers Compartment Validation & Spatial Analysis

## 1. High-Level Compartment Mapping (Dotplot & UMAP Feature Results)

The broad compartment marker panel successfully validated the global UMAP topology across the 22 Leiden clusters, resolving the dataset into 5 major anatomical super-clusters:

* **Myeloid Super-Cluster (Bottom Right):** Expresses high pan-immune markers (`PTPRC`, `SRGN`) alongside lineage-specific myeloid markers (`ITGAM`, `LYZ`, `AIF1`, `CSF1R`). Encompasses alveolar macrophages, interstitial macrophages, monocytes, and dendritic cells.
* **Lymphoid Super-Cluster (Top Right):** Positively labeled by pan-immune markers (`PTPRC`, `SRGN`) and lymphoid-specific expression (`NCAM1`, `CD3D`). Encompasses T cells, NK cells, and B/Plasma cell lineages.
* **Epithelial Super-Cluster (Far Left):** Completely negative for pan-immune markers (`PTPRC`- / `SRGN`-), showing exclusive expression of structural epithelial markers (`EPCAM`, `KRT8`, `KRT18`). Encompasses AT1, AT2, Ciliated, and Club cells.
* **Stromal / Fibroblast Island (Bottom Left):** Negative for immune and epithelial markers, displaying localized expression of extracellular matrix genes (`COL1A1`, `DCN`, `LUM`). Encompasses structural fibroblasts, pericytes, and smooth muscle cells.
* **Endothelial Island (Bottom Center / Periphery):** Isolated cluster displaying specific expression of vascular markers (`PECAM1`). Encompasses vascular and lymphatic endothelial cells.

---

## 2. Biological Interpretation of Pan-Immune Lineages

* **Pan-Immune Co-activation (`PTPRC` / `SRGN`):** The pan-immune panel correctly highlights both the **Myeloid** (Bottom Right) and **Lymphoid** (Top Right) super-clusters simultaneously. Because both lineages originate from hematopoietic stem cells, co-expression of `PTPRC` (CD45) and `SRGN` across both islands confirms biological accuracy.
* **Specificity & Negative Controls:** The complete absence of `PTPRC` and `SRGN` signal within the Epithelial, Stromal, and Endothelial islands demonstrates clean normalization and effective scVI denoising, establishing well-defined boundary conditions for fine-grained subtype annotation.

---

## 3. Recommended Subtype Marker Reference Panel (ABOVE in DICTIONARY at begining of this section)

plot_scvi_markers(
    genes=covid_lung_markers["Pan-Immune"],
    model=model_combined,
    adata=umap_adata,
    ncols=3,
    size=8,
)

Pan_Immune = ["PTPRC", "LAPTM5", "SRGN"]
plot_scvi_dotplot(
    genes=Pan_Immune,
    model=model_combined,
    adata=umap_adata,
    groupby="leiden",
)

$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$

# 6.5.3 SEARCHING...  MYELOID SUBTYPES:

* "Alveolar_Macs": ["FABP4", "MARCO"],
* "Interstitial_Macs": ["CD163", "MRC1", "C1QA"],
* "Monocytes_Classical": ["CD14", "S100A8", "S100A9"],
* "Monocytes_NonClassical": ["FCGR3A", "CX3CR1"],
* "Dendritic_cDC1": ["CLEC9A", "XCR1"],
* "Dendritic_cDC2": ["CD1C", "FCER1A"],
* "pDCs": ["LILRA4", "CLEC4C"],
* "Neutrophils": ["FCGR3B", "CSF3R"],

$$$$


In [ ]:
import json

myeloids = [
    "Alveolar_Macs",
    "Interstitial_Macs",
    "Monocytes_general",
    "Monocytes_Classical",
    "Monocytes_NonClassical",
    "Dendritic_cells_gen",
    "Dendritic_cDC1",
    "Dendritic_cDC2",
    "pDCs",
    "Neutrophils",
    "Mast_Cells",
]
markers_tp = myeloids

# 1. Extract sub_markers dictionary filtered by existing genes in var_names
sub_markers = {
    cat: [g for g in covid_lung_markers[cat] if g in umap_adata.var_names]
    for cat in markers_tp
    if cat in covid_lung_markers
}

# 2. Print category breakdown
print("Testing MYELOID SUBTYPES (Filtered for dataset genes):")
print(json.dumps(sub_markers, indent=4))

# 3. Report missing genes per category if any were dropped
missing_summary = {}
for cat in markers_tp:
    if cat in covid_lung_markers:
        dropped = set(covid_lung_markers[cat]) - set(sub_markers[cat])
        if dropped:
            missing_summary[cat] = list(dropped)

if missing_summary:
    print("\n⚠️ Skipped missing genes by category:")
    print(json.dumps(missing_summary, indent=4))

# 4. Extract unique flattened list of genes to plot
genes_to_plot = list(dict.fromkeys([gene for sublist in sub_markers.values() for gene in sublist]))

print(f"\nPlotting {len(genes_to_plot)} valid genes...")

# 5. Run UMAP plot
plot_scvi_markers(
    genes=genes_to_plot,
    model=model_combined,
    adata=umap_adata,
    ncols=3,
    cmap="magma",
    size=8,
)

# 6. Run Dotplot
plot_scvi_dotplot(
    genes=genes_to_plot,
    model=model_combined,
    adata=umap_adata,
    groupby="leiden",
)


$$$$

## ANALYSIS MYELOIDS:



In [ ]:
sc.pl.umap(
    umap_adata,
    color=["leiden"],
    legend_loc="on data",
    legend_fontsize=10,
    title=f"Leiden Clustering (Resolution {test_res})"
)

In [ ]:
## Confirming Dendritic Cell (DC) Sub-populations across Leiden clusters

# Look at the top differentially expressed markers for Cluster 16
markers_scanpy_filtered[markers_scanpy_filtered['group'] == '16'].head(10)

In [ ]:
# Query top marker genes for remaining small immune clusters
markers_scanpy_filtered[markers_scanpy_filtered['group'].isin(['14', '18'])].groupby('group').head(5)

# 
#### Cluster 14 $\rightarrow$ Smooth Muscle / Pericytes: 
## * Evidence: Driven by CALD1 (Caldesmon 1), PRKG1, COL4A1 (type IV collagen), and RYR2. These reflect contractile vascular/mural cells located adjacent to your fibroblast super-cluster (Cluster 4).   

#### Cluster 18 $\rightarrow$ Airway Secretory / Epithelial: 
##  * Evidence: Driven by SEMA3A, ST6GALNAC3, and PPFIBP1, sitting right between the main epithelial body (Clusters 1 & 2) and the endothelial island. 

In [ ]:

markers_scanpy_filtered[markers_scanpy_filtered.names == 'MARCO']

## ANALYSIS:

#### 1. Cluster 17 $\rightarrow$ Neutrophils
   * Evidence: CSF3R shows an extremely strong, exclusive signal specifically covering Cluster 17 on the far left.   

#### 2. Cluster 10 $\rightarrow$ Classical Monocytes
    * Evidence: Look at the bottom-most pink satellite of the myeloid island in the Leiden UMAP (Cluster 10). That exact position corresponds to the bright expression of S100A8 and S100A9.
      
#### 3. Cluster 12 & 19 $\rightarrow$ Classical Monocytes / Monocyte-Derived Macrophages
    * Evidence: These clusters lie directly on the bottom border of Cluster 0. They show strong CD14 expression alongside C1QA, representing newly recruited monocytes transitioning into macrophages.

#### 4. Cluster 15  $\rightarrow$ Non-Classical Monocytes 
    * Evidence: The rightmost tail extension (Cluster 15 and cluster 16) aligns with the localized expression of FCGR3A (CD16). (Note: Test CD1C / LILRA4 in your dotplot to confirm if 16 is a DC sub-population).
      
#### 5. Cluster 21 (Top rim of Cluster 0) $\rightarrow$ Residual Alveolar Macrophages
    * Evidence: MARCO shows a distinct localized expression patch at the very top edge of Cluster 0 (where Cluster 21 sits).

#### 6. CLUSTER 16


#### Cluster 14 $\rightarrow$ Smooth Muscle / Pericytes: 

    * Evidence: Driven by CALD1 (Caldesmon 1), PRKG1, COL4A1 (type IV collagen), and RYR2. These reflect contractile vascular/mural cells located adjacent to your fibroblast super-cluster (Cluster 4).   

 #### Cluster 18 $\rightarrow$ Airway Secretory / Epithelial: 
    * Evidence: Driven by SEMA3A, ST6GALNAC3, and PPFIBP1, sitting right between the main epithelial body (Clusters 1 & 2) and the endothelial island.   

$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$

# 6.5.4 SEARCHING...  Lymphoid Subtypes
    "T_Cells_General": ["CD3D", "CD3E"],
    "T_CD4": ["CD4"],
    "T_CD8": ["CD8A", "CD8B"],
    "NK_Cells": ["NCAM1", "NKG7", "GNLY"],
    "B_Cells": ["MS4A1", "CD19"],
    "Plasma_Cells": ["SDC1", "MZB1"],
$$$$

In [ ]:
import json

# 1. Select ONLY the general broad categories
lymphoids = [
     "T_Cells_General",        # TRAC = T-cell receptor alpha constant region
    "T_CD4_Helper",                                    # MHC-II restricted helper/regulatory T cells
    "T_CD8_Cytotoxic",                        # MHC-I restricted cytotoxic T lymphocytes (CTLs)
    "NK_Cells",     # NCAM1 = CD56; KLRD1 = CD94; TRDC = Gamma/delta T-cell subset overlap
    "B_Cells",               # MS4A1 = CD20; CD79A = Ig-alpha signaling subunit
    "Plasma_Cells",       # SDC1 = CD138; TNFRSF17 = BCMA (Antibody-secreting terminally differentiated B cells)

]
import json

markers_tp = lymphoids

# 1. Extract sub_markers dictionary filtered by existing genes in var_names
sub_markers = {
    cat: [g for g in covid_lung_markers[cat] if g in umap_adata.var_names]
    for cat in markers_tp
    if cat in covid_lung_markers
}

# 2. Print category breakdown
print("Testing Lymphoid SUBTYPES (Filtered for dataset genes):")
print(json.dumps(sub_markers, indent=4))

# 3. Report missing genes per category if any were dropped
missing_summary = {}
for cat in markers_tp:
    if cat in covid_lung_markers:
        dropped = set(covid_lung_markers[cat]) - set(sub_markers[cat])
        if dropped:
            missing_summary[cat] = list(dropped)

if missing_summary:
    print("\n⚠️ Skipped missing genes by category:")
    print(json.dumps(missing_summary, indent=4))

# 4. Extract unique flattened list of genes to plot
genes_to_plot = list(dict.fromkeys([gene for sublist in sub_markers.values() for gene in sublist]))

print(f"\nPlotting {len(genes_to_plot)} valid genes...")

# 5. Run UMAP plot
plot_scvi_markers(
    genes=genes_to_plot,
    model=model_combined,
    adata=umap_adata,
    ncols=3,
    cmap="magma",
    size=8,
)

# 6. Run Dotplot
plot_scvi_dotplot(
    genes=genes_to_plot,
    model=model_combined,
    adata=umap_adata,
    groupby="leiden",
)

In [ ]:
sc.pl.umap(
    umap_adata,
    color=["leiden"],
    legend_loc="on data",
    legend_fontsize=10,
    title=f"Leiden Clustering (Resolution {test_res})"
)

$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$

# # 6.5.5 SEARCHING... Epithelial Subtypes
    "AT1_Cells": ["AGER", "PDPN"],
    "AT2_Cells": ["SFTPC", "SFTPA1"],
    "Ciliated": ["FOXJ1", "TPPP3"],
    "Secretory_Club": ["SCGB1A1", "MUC5AC"],
    

In [ ]:
import json

# 1. Select ONLY the general broad categories
epithel = [
    "AT1_Cells",
    "AT2_Cells",
    "Ciliated",
    "Secretory_Club",
     "Airway_Goblet",
    "Airway_endothelial"
]
markers_tp = epithel

# 1. Extract sub_markers dictionary filtered by existing genes in var_names
sub_markers = {
    cat: [g for g in covid_lung_markers[cat] if g in umap_adata.var_names]
    for cat in markers_tp
    if cat in covid_lung_markers
}

# 2. Print category breakdown
print("Testing EPITHELIAL SUBTYPES (Filtered for dataset genes):")
print(json.dumps(sub_markers, indent=4))

# 3. Report missing genes per category if any were dropped
missing_summary = {}
for cat in markers_tp:
    if cat in covid_lung_markers:
        dropped = set(covid_lung_markers[cat]) - set(sub_markers[cat])
        if dropped:
            missing_summary[cat] = list(dropped)

if missing_summary:
    print("\n⚠️ Skipped missing genes by category:")
    print(json.dumps(missing_summary, indent=4))

# 4. Extract unique flattened list of genes to plot
genes_to_plot = list(dict.fromkeys([gene for sublist in sub_markers.values() for gene in sublist]))

print(f"\nPlotting {len(genes_to_plot)} valid genes...")

# 5. Run UMAP plot
plot_scvi_markers(
    genes=genes_to_plot,
    model=model_combined,
    adata=umap_adata,
    ncols=3,
    cmap="magma",
    size=8,
)

# 6. Run Dotplot
plot_scvi_dotplot(
    genes=genes_to_plot,
    model=model_combined,
    adata=umap_adata,
    groupby="leiden",
)


In [ ]:
sc.pl.umap(
    umap_adata,
    color=["leiden"],
    legend_loc="on data",
    legend_fontsize=10,
    title=f"Leiden Clustering (Resolution {test_res})"
)

$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$

--------------
---------------
# # 6.5.6 SEARCHING... Stromal & Endothelial Subtypes
    --- Stromal & Endothelial Subtypes ---
    "Endothelial_general": ["PECAM1", "EGFL7", "ID3"],          # Pan-endothelial nuclear and cell-surface identifiers
    "Endothelial_Vascular": ["PECAM1", "VWF", "CD31"],          # PECAM1 = CD31; VWF = Blood vessel/capillary lineage
    "Endothelial_Lymphatic": ["PROX1", "LYVE1"],                # PROX1 = Master lymphatic TF; LYVE1 = Lymphatic vessel endothelial hyaluronan receptor
    "Fibroblasts": ["COL1A1", "DCN", "COL6A2", "VTN", "LUM"],   # DCN = Decorin; LUM = Lumican (Interstitial lung fibroblasts)
    "Smooth_Muscle": ["ACTA2", "MYL9", "PDGFRB"],               # ACTA2 = Alpha-smooth muscle actin (aSMA); Contractile vascular/airway smooth muscle
    "Pericytes": ["PDGFRB", "RERGL", "CSPG4"]                   # PDGFRB = CD140b; CSPG4 = NG2 (Mural microvascular support cells)
}

--------------
---------------

In [ ]:
import json

# 1. Select ONLY the general broad categories
endo = [
    "Endothelial_general",
    "Endothelial_Vascular",
    "Endothelial_Lymphatic",
    "Fibroblasts",
    "Smooth_Muscle",
    "Pericytes"
]
markers_tp = endo

# 1. Extract sub_markers dictionary filtered by existing genes in var_names
sub_markers = {
    cat: [g for g in covid_lung_markers[cat] if g in umap_adata.var_names]
    for cat in markers_tp
    if cat in covid_lung_markers
}

# 2. Print category breakdown
print("Testing EndoTHELIAL SUBTYPES (Filtered for dataset genes):")
print(json.dumps(sub_markers, indent=4))

# 3. Report missing genes per category if any were dropped
missing_summary = {}
for cat in markers_tp:
    if cat in covid_lung_markers:
        dropped = set(covid_lung_markers[cat]) - set(sub_markers[cat])
        if dropped:
            missing_summary[cat] = list(dropped)

if missing_summary:
    print("\n⚠️ Skipped missing genes by category:")
    print(json.dumps(missing_summary, indent=4))

# 4. Extract unique flattened list of genes to plot
genes_to_plot = list(dict.fromkeys([gene for sublist in sub_markers.values() for gene in sublist]))

print(f"\nPlotting {len(genes_to_plot)} valid genes...")

# 5. Run UMAP plot
plot_scvi_markers(
    genes=genes_to_plot,
    model=model_combined,
    adata=umap_adata,
    ncols=3,
    cmap="magma",
    size=8,
)

# 6. Run Dotplot
plot_scvi_dotplot(
    genes=genes_to_plot,
    model=model_combined,
    adata=umap_adata,
    groupby="leiden",
)

In [ ]:
sc.pl.umap(
    umap_adata,
    color=["leiden"],
    legend_loc="on data",
    legend_fontsize=10,
    title=f"Leiden Clustering (Resolution {test_res})"
)

# # 6.5.7 WHEN YOU CAN'T IDENTIFY A CLUSTER...

#### CLUSTER 6

In [ ]:
# STEP 1
# get a list of the top genes in that cluster (alreadt sorted for log2FC):


import pandas as pd

# Filter for cluster '6', positive fold-change, and significant adjusted p-value
c6_distinguishing = markers_scanpy_filtered[
    (markers_scanpy_filtered['group'] == '6') & 
    (markers_scanpy_filtered['logfoldchanges'] > 0) & 
    (markers_scanpy_filtered['pvals_adj'] < 0.05)
].sort_values(by='logfoldchanges', ascending=False)

# Get top 30 distinguishing genes
top30_c6_genes = c6_distinguishing['names'].head(30).tolist()

print(f"Top {len(top30_c6_genes)} marker genes distinguishing Cluster 6 from all others:")
for rank, gene in enumerate(top30_c6_genes, start=1):
    lfc = c6_distinguishing.loc[c6_distinguishing['names'] == gene, 'logfoldchanges'].values[0]
    print(f"{rank:2d}. {gene:<12} (log2FC: {lfc:.2f})")
    
#Step 2: Past the highest genes in pangloadb.se to check to which cells it might belong
    
#STEP 3: If step 2 is not enough use an annotation tool to paste the whole list of genes and get the cellc andidates

# * https://davidbioinformatics.nih.gov/ --- THEN --- Funcitonal annotation (select correct species) -- THEN -- Funcional annotation chart/table

In [ ]:
# Test stromal vs mural candidate markers on Cluster 6
c6_validation_genes = [
    # Top candidates from your output
    "ROBO2", "SLIT2", "SLIT3", "ITGA8", "ELN", "TNXB", "LAMA2", "CACNA1C",
    # Reference markers from COVID_LUNG_MARKERS for comparison
    "COL1A1", "DCN", "LUM",       # Fibroblasts
    "ACTA2", "MYL9", "PDGFRB",    # Smooth Muscle / Pericytes
    "CSPG4", "VTN"
]

plot_scvi_dotplot(
    genes=c6_validation_genes,
    model=model_combined,
    adata=umap_adata,
    groupby="leiden",
)

#If COL1A1, DCN, LUM, ELN, and SLIT2 co-express strongly: Fibroblasts / Adventitial Fibroblasts.

#If ACTA2, MYL9, PDGFRB, ITGA8, and CACNA1C co-express strongly: Smooth Muscle / Pericytes.

Cluster 6 (Adventitial / Interstitial Fibroblasts):
* Strong positive markers: Shows dark red, high-fraction expression exclusively across ROBO2, SLIT2, SLIT3, ITGA8, ELN, TNXB, LAMA2, and CACNA1C.
* Low/Moderate Core ECM: Expresses moderate levels of general pan-fibroblast matrix genes (COL1A1, DCN, LUM) relative to Cluster 4
* Negative for Mural markers: Lacks expression of contractile smooth muscle / pericyte markers (ACTA2, MYL9, PDGFRB, CSPG4).

In [ ]:
### CLUSTER 11

In [ ]:
distinguishing = markers_scanpy_filtered[
    (markers_scanpy_filtered['group'] == '11') & 
    (markers_scanpy_filtered['logfoldchanges'] > 0) & 
    (markers_scanpy_filtered['pvals_adj'] < 0.05)
].sort_values(by='logfoldchanges', ascending=False)

# Get top 30 distinguishing genes
top30_c_genes = distinguishing['names'].head(30).tolist()

print(f"Top {len(top30_c_genes)} marker genes distinguishing Cluster 11 from all others:")
for rank, gene in enumerate(top30_c_genes, start=1):
    lfc = distinguishing.loc[distinguishing['names'] == gene, 'logfoldchanges'].values[0]
    print(f"{rank:2d}. {gene:<12} (log2FC: {lfc:.2f})")
    

In [ ]:
c11_validation_genes = [
    # Top candidates from your output
    "SCGB3A1", "SCGB1A1", "SLPI", "WFDC2", "MUC4", "PIGR", "KRT19", "EHF",
    # Reference panel markers for comparison
    "EPCAM", "KRT8", "KRT18",      # Broad Epithelial
    "MUC5AC",                       # Goblet / Mucinous Secretory
    "FOXJ1", "TPPP3",               # Ciliated
    "SFTPC", "AGER"                 # AT2 / AT1
]

plot_scvi_dotplot(
    genes=c11_validation_genes,
    model=model_combined,
    adata=umap_adata,
    groupby="leiden",
)

### Cluster 11 (Airway Secretory / Club Cells):
* Strong positive drivers: Shows intense, high-fraction expression of secretoglobins (SCGB3A1, SCGB1A1), protective mucosal factors (SLPI, WFDC2, MUC4), polymeric Ig receptor (PIGR), epithelial cytokeratins (KRT19, KRT8, KRT18), and general epithelial drivers (EHF, EPCAM).
* Negative for specialized sublineages: Lacks expression of ciliated markers (TPPP3), AT2 markers (SFTPC), and AT1 markers (AGER).

### * Cluster 17:
* expressed TPPP3 strongly, confirming its identity as Ciliated epithelial cells. Neutrophils typically express high FCGR3B, CSF3R, and S100A8 without cytokeratins or ciliated markers.

$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$

--------------
---------------
# 6.6

--------------
---------------

In [ ]:
import pandas as pd
from markers import CELL_TYPE

# Convert dictionary into a structured DataFrame
df_cell_types = pd.DataFrame(
    list(CELL_TYPE.items()), 
    columns=["Cluster", "Cell Type Annotation"]
)

# Display in Jupyter Notebook
display(df_cell_types)

---------
### 6.3 SAVING MARKER LABELS
---------

In [ ]:
umap_adata

# SAVING MARKER LABELS¶ (NEW VERSION - check)

In [ ]:
# 1. Save the DE dataframes inside .uns (matching the tutorial)
umap_adata.uns["scvi_markers"] = markers_scvi_filtered
umap_adata.uns["scanpy_markers"] = markers_scanpy_filtered

# 2. Save your cluster annotation dictionary in .uns
umap_adata.uns["cell_type_dict"] = CELL_TYPE

# 3. Map cell labels into .obs for plotting & downstream analysis
umap_adata.obs["cell_type"] = umap_adata.obs["leiden"].map(CELL_TYPE)


# 4. Save the annotated AnnData object
umap_adata.write_h5ad("manual_labeled.h5ad")

In [ ]:
# 5. Save the trained scVI model
model_dir = "scvi_model_combined"
model_combined.save(model_dir, overwrite=True)

In [ ]:
# 4. Plot higher-resolution, larger UMAP
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12, 10), dpi=150)

sc.pl.umap(
    umap_adata,
    color=["cell_type"],
    legend_loc="on data",
    legend_fontsize=9,
    legend_fontweight="bold",
    title="COVID-19 Lung Cell Annotations",
    ax=ax,
    show=False,
)

plt.tight_layout()
plt.show()